In [10]:
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper import RAGBase
from ingest import load_data, build_index

In [11]:
load_dotenv()
openai_client = OpenAI()

In [12]:
data = load_data()
index = build_index(data)

In [13]:
instructions = """
You are learning assistant.
Your task is to answer questions based on the provided context.
Use the context to find relevant information and provide accurate answers.
Use only the facts from the CONTEXT when answering the QUESTION.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [14]:
assistant.rag('What is regularization')

'Regularization is a technique used to avoid overfitting by making the model more simple.\n\nIt helps reduce the complexity of a model, such as a neural network, by reducing the magnitude of weights and biases. Smaller weights can make some neurons negligible, which makes the network less complex and less prone to overfitting.\n\nWould you like to explore L1 vs L2 regularization or other related topics?'

In [15]:
assistant.rag('What is reulaization')

'It looks like you mean **regularization**.\n\n**Regularization** is a technique used in machine learning to reduce overfitting and improve a model’s performance on new data. It does this by discouraging the model from becoming too complex.\n\nIf you meant something else by “reulaization,” let me know.\n\nWould you like to explore related topics like **overfitting**, **underfitting**, or specific regularization methods?'

In [16]:
messages = [
    {"role": "user", "content": "What clustering algorithms are described in the ds_qa database?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

print(response.output_text)

I don’t have direct access to the `ds_qa` database contents from here, so I can’t reliably name the clustering algorithms it describes without seeing the relevant records or schema.

If you want, I can help you find them quickly. For example, you could query the database for terms like:

- `clustering`
- `k-means`
- `hierarchical clustering`
- `DBSCAN`
- `Gaussian Mixture Model`
- `spectral clustering`
- `mean shift`
- `BIRCH`
- `affinity propagation`

If you can share:
1. the table/schema, or  
2. a dump/export of the relevant rows, or  
3. the result of a search query,

I can extract and summarize the clustering algorithms described in `ds_qa` for you.


In [17]:
def search(query):
    boost_dict = {"question": 3.0}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [18]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the ds_qa database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the ds_qa database."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [19]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
#    tool_choice="required"
)

print(response.output)

[ResponseFunctionToolCall(arguments='{"query":"clustering algorithms"}', call_id='call_4sBYiIIzpHqjowLVoZk6WdtF', name='search', type='function_call', id='fc_0808821df7c2b30b006a3e87b0ad04819f9b35b5cd3bea7acf', namespace=None, status='completed')]


In [20]:
import json


call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [21]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [22]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

The ds_qa database describes these clustering algorithms:

1. **K-Means**  
   - A well-known clustering algorithm, noted for being easy to interpret and implement.

2. **MiniBatch KMeans**  
   - A scalable version of K-Means that updates centroids using mini-batches, making it suitable for very large datasets.

3. **BIRCH**  
   - Balanced Iterative Reducing and Clustering using Hierarchies, which first creates a compact summary of the dataset and then clusters that summary.

If you want, I can also summarize the **clustering evaluation metrics** mentioned in ds_qa, like **Silhouette coefficient** and **Dunn’s Index**.


In [23]:
usage = response.usage
input_tokens, output_tokens = usage.input_tokens, usage.output_tokens

In [24]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(input_tokens, output_tokens)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.00025605


The logic behind:
1. Make a call to the LLM <-- we pay
2. LLM decides to call the search tool ('params')
3. We invoke the search, we get the results
4. Send the request back to the LLM (another call) <-- we pay
5. LLM processes the results
6a. LLM gives the answer

6b. LLM decides to make another tool call
7. Send one more API request <-- we pay

In [5]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        'type': 'function_call_output',
        'call_id': call.call_id,
        'output': result_json
    }

In [35]:
question = 'What clustering methods are in the ds_qa database?'

In [36]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}')
    has_function_call = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool],
        tool_choice='required'
    )

    messages.extend(response.output)
        
    for item in response.output:

        if item.type == 'function_call':
            print(f'function call: {item.name}, {item.arguments}')
            call_output = make_call(item)
            messages.append(call_output)
            has_function_call = True
        
        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)

    it += 1
    if has_function_call == False or it > 5:
        break

iteration #1


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.4-mini in organization org-gsXe3AZppEQg3VfDxCS4SYvQ on tokens per min (TPM): Limit 100000, Used 100000, Requested 777. Please try again in 5h35m39.84s. Visit https://platform.openai.com/account/rate-limits to learn more. You can increase your rate limit by adding a payment method to your account at https://platform.openai.com/account/billing.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

In [31]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"reinforcement learning definition"}', call_id='call_DTcGjYa42YZM8TMLBroWgUqN', name='search', type='function_call', id='fc_04a560c30f11b1c6006a3e87bc5f0881a0947597bcb628e91c', namespace=None, status='completed')]